# MLB intro — sportsdataverse-py

Baseball from three sources: the **MLB Stats API** (`mlb_api_*`, backed by statsapi.mlb.com), **Statcast** pitch-level data (`statcast_*`, from Baseball Savant), and **ESPN MLB** (`espn_mlb_*`).

R companion: [baseballr](https://billpetti.github.io/baseballr/). Python neighbors: [pybaseball](https://github.com/jldbc/pybaseball), [MLB-StatsAPI](https://github.com/toddrob99/MLB-StatsAPI). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse.mlb as mlb

# 1. MLB Stats API (`mlb_api_*`)

The official MLB Stats API at `statsapi.mlb.com`. These wrappers return the raw JSON as a `dict` by default; many have a paired `parse_mlb_api_*()` helper (or a `return_parsed=True` flag) that flattens the JSON into a tidy `clean_names` polars frame.

## Teams

`mlb_api_teams()` returns a `dict`; `parse_mlb_api_teams()` flattens it to one row per club.

In [ ]:
teams = mlb.parse_mlb_api_teams(mlb.mlb_api_teams(season=2024))
teams.select(['id', 'name', 'abbreviation', 'location_name', 'team_name']).head()

## Schedule

`mlb_api_schedule(date=...)` accepts a single `YYYY-MM-DD` date (or `start_date`/`end_date`, `team_id`, `season`). `parse_mlb_api_schedule()` gives one row per game, including each game's `game_pk` — the id you feed to the boxscore / play-by-play endpoints.

In [ ]:
schedule = mlb.parse_mlb_api_schedule(mlb.mlb_api_schedule(date='2024-07-01'))
schedule.select([
    'game_pk', 'status_detailed_state',
    'teams_away_team_name', 'teams_away_score',
    'teams_home_team_name', 'teams_home_score',
]).head()

## Standings

`mlb_api_standings(season=...)` covers both leagues by default (`league_id='103,104'`). `parse_mlb_api_standings()` returns one row per team with wins/losses, division rank, and run differential.

In [ ]:
standings = mlb.parse_mlb_api_standings(mlb.mlb_api_standings(season=2024))
(standings
    .select(['team_name', 'standings_division_name', 'wins', 'losses', 'winning_percentage', 'division_rank'])
    .sort('wins', descending=True)
    .head(10))

## Team roster

`mlb_api_team_roster(team_id=..., season=...)` returns a tidy frame directly (one row per player). Here, the 2024 New York Yankees (`team_id=147`).

In [ ]:
roster = mlb.mlb_api_team_roster(team_id=147, season=2024)
roster.select(['jersey_number', 'person_id', 'person_full_name', 'position_abbreviation', 'status_description']).head()

## Player bio & season stats

`mlb_api_person(person_id=...)` returns a one-row bio frame. `mlb_api_person_stats(...)` returns a `dict`; `parse_mlb_api_person_stats()` flattens the stat splits. Below: Aaron Judge (`person_id=592450`).

In [ ]:
judge = mlb.mlb_api_person(person_id=592450)
judge.select(['id', 'full_name', 'primary_number', 'birth_date', 'height', 'weight', 'mlb_debut_date'])

In [ ]:
stats = mlb.parse_mlb_api_person_stats(
    mlb.mlb_api_person_stats(person_id=592450, stats='season', group='hitting', season=2024)
)
stats.select(['season', 'stat_games_played', 'stat_home_runs', 'stat_rbi', 'stat_avg', 'stat_obp', 'stat_slg', 'stat_ops'])

## Boxscore

`mlb_api_boxscore(game_pk=...)` returns the full boxscore. Use `return_parsed=False` to get the raw `dict`, which carries per-team batting/pitching lines under `teams.home`/`teams.away`. Game `744914` is the Astros @ Blue Jays from the schedule above.

In [ ]:
box = mlb.mlb_api_boxscore(game_pk=744914, return_parsed=False)
home = box['teams']['home']
team_batting = home['teamStats']['batting']
{
    'team': home['team']['name'],
    'runs': team_batting['runs'],
    'hits': team_batting['hits'],
    'home_runs': team_batting['homeRuns'],
    'avg': team_batting['avg'],
    'rbi': team_batting['rbi'],
}

## Play-by-play

`mlb_api_play_by_play(game_pk=..., return_parsed=False)` returns a `dict` with an `allPlays` list — one entry per plate appearance. Flatten it with `pl.json_normalize` to get a frame whose columns use dot-notation (`result.event`, `about.inning`, `matchup.batter.fullName`).

In [ ]:
raw_pbp = mlb.mlb_api_play_by_play(game_pk=744914, return_parsed=False)
plays = pl.json_normalize(raw_pbp['allPlays'], separator='.', max_level=2)
plays.select([
    'about.inning', 'about.halfInning',
    'matchup.batter.fullName', 'matchup.pitcher.fullName',
    'result.event', 'result.description',
]).head()

In [ ]:
# Count plate-appearance outcomes
(plays
    .group_by('result.event')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .head(10))

# 2. Statcast (`statcast_*`)

Pitch-level tracking data from [Baseball Savant](https://baseballsavant.mlb.com/). Keep queries **small** (one player, one game, or a 1–2 day window) — a full season is millions of pitches.

## Pitch-level search

`statcast_search(start_date=, end_date=, batters_lookup=)` pulls every pitch matching the filter. Here: every pitch Aaron Judge saw over a 2-day window. Each row is a single pitch with 100+ columns (velocity, spin, launch metrics, expected stats).

In [ ]:
pitches = mlb.statcast_search(
    start_date='2024-07-01',
    end_date='2024-07-02',
    batters_lookup=592450,
)
print(pitches.shape)
pitches.select([
    'game_date', 'player_name', 'pitch_type', 'release_speed',
    'launch_speed', 'launch_angle', 'events', 'description',
]).head()

In [ ]:
# Pitch mix Judge faced, with average velocity
(pitches
    .filter(pl.col('pitch_type').is_not_null())
    .group_by('pitch_type')
    .agg([
        pl.len().alias('pitches'),
        pl.col('release_speed').mean().round(1).alias('avg_velo'),
    ])
    .sort('pitches', descending=True))

## Single-game feed

`statcast_gamefeed(game_pk=...)` returns a `dict` of the Savant game feed — scoreboard, per-team batters/pitchers, and at-bat detail for one game.

In [ ]:
feed = mlb.statcast_gamefeed(game_pk=744914)
{k: feed[k] for k in ['game_status', 'gameDate', 'team_home', 'team_away']}

## Leaderboards

The `statcast_leaderboard_*` family wraps Savant's pre-aggregated season leaderboards — fast, since the heavy lifting happens server-side. Here: 2024 sprint speed.

In [ ]:
sprint = mlb.statcast_leaderboard_sprint_speed(year=2024, min_opp=10)
(sprint
    .select(['last_name, first_name', 'team', 'position', 'competitive_runs', 'sprint_speed'])
    .sort('sprint_speed', descending=True)
    .head(10))

# 3. ESPN MLB (`espn_mlb_*`)

Same ESPN conventions as the other leagues: schedule/teams return wide polars frames; play-by-play returns a `dict` whose `plays` key is a list of raw dicts; scores are **strings**.

## Teams

In [ ]:
espn_teams = mlb.espn_mlb_teams()
espn_teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation', 'team_display_name']).head()

## Schedule

`espn_mlb_schedule(dates=YYYYMMDD)` returns one row per game with `home_display_name`/`away_display_name`. The `home_score`/`away_score` columns are **strings** — cast before doing arithmetic.

In [ ]:
espn_sched = mlb.espn_mlb_schedule(dates=20240701)
espn_sched.select([
    'game_id', 'away_display_name', 'away_score',
    'home_display_name', 'home_score', 'status_type_completed',
]).head()

## Standings

In [ ]:
espn_standings = mlb.espn_mlb_standings(season=2024)
(espn_standings
    .select(['team_display_name', 'group_name', 'wins', 'losses', 'games_behind', 'streak'])
    .sort('wins', descending=True)
    .head(10))

## Play-by-play

`espn_mlb_pbp(game_id=...)` returns a `dict`. `pbp['plays']` is a list of raw dicts; build a frame with `pl.DataFrame(..., infer_schema_length=None)`. ESPN nests `period` and `type` as structs — reach into them with `.struct.field()`.

In [ ]:
espn_pbp = mlb.espn_mlb_pbp(game_id=401569738)
espn_plays = pl.DataFrame(espn_pbp['plays'], infer_schema_length=None)
espn_plays.select([
    pl.col('period').struct.field('number').alias('inning'),
    pl.col('text'),
    pl.col('scoringPlay'),
    pl.col('awayScore'),
    pl.col('homeScore'),
]).head()

In [ ]:
# Just the scoring plays
(espn_plays
    .filter(pl.col('scoringPlay'))
    .select([
        pl.col('period').struct.field('number').alias('inning'),
        pl.col('text'),
        pl.col('awayScore'),
        pl.col('homeScore'),
    ]))

## Pipeline example: cross-source HR check

Combine two sources for one player. Pull Aaron Judge's 2024 season home-run total from the MLB Stats API, then confirm the batted-ball event types Statcast recorded for him over the same 2-day window line up with the play-by-play.

In [ ]:
season_hr = (stats
    .select(pl.col('stat_home_runs').alias('season_home_runs'))
    .with_columns(pl.lit('Aaron Judge').alias('player'),
                  pl.lit(2024).alias('season')))

statcast_events = (pitches
    .filter(pl.col('events').is_not_null())
    .group_by('events')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True))

print(season_hr)
statcast_events

## Cross-references

- R companion: [baseballr](https://billpetti.github.io/baseballr/)
- Data sources: MLB Stats API (`statsapi.mlb.com`), Statcast / Baseball Savant, ESPN MLB API
- Python neighbors: [pybaseball](https://github.com/jldbc/pybaseball), [MLB-StatsAPI](https://github.com/toddrob99/MLB-StatsAPI)
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: `docs/docs/mlb/index.md`
- Browse the cross-sport overview in `01_quickstart.ipynb`, or compare conventions with the other league intros (`04_nba_intro.ipynb`, `07_nhl_intro.ipynb`).